In [ ]:
import sys
import os
import importlib

from tabulate import tabulate
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append("../src")

import utils
import plot
import rstats

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(plot)
importlib.reload(rstats)

# Run before setting latex params, else it's overridden
sns.set_style("whitegrid")
sns.set_context("notebook")

# FIXME: must have latex installed in this path ("/Library/TeX/texbin")
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Computer Modern Roman"]
plt.rcParams["figure.dpi"] = 300

base = 12
plt.rcParams.update(
    {
        "font.size": base,
        "axes.titlesize": base + 3,
        "axes.labelsize": base + 2,
        "xtick.labelsize": base,
        "ytick.labelsize": base,
        "legend.fontsize": base,
        "figure.titlesize": base + 5,
    }
)

In [ ]:
files = ["parkinson", "swear-fluency", "italian", "german"]
metrics = ["d_next", "vel", "acc", "entropy", "d_centroid"]

# Precompute correlations for each file
corrs = []
for FILENAME in files:
    print(f"{FILENAME=}")

    src = f"../results/openai/{FILENAME}/metrics.csv"
    df1 = utils.load(src)[metrics].reset_index(drop=True)

    src = f"../results/gemini/{FILENAME}/metrics.csv"
    df2 = utils.load(src)[metrics].reset_index(drop=True)

    src = f"../results/qwen/{FILENAME}/metrics.csv"
    df3 = utils.load(src)[metrics].reset_index(drop=True)

    src = f"../results/fasttext/{FILENAME}/metrics.csv"
    df4 = utils.load(src)[metrics].reset_index(drop=True)

    # Align rows
    df1, df2 = df1.align(df2, join="inner", axis=0)
    df1, df3 = df1.align(df3, join="inner", axis=0)
    df1, df4 = df1.align(df4, join="inner", axis=0)

    combined = df1.add_suffix("_m1").join(
        [df2.add_suffix("_m2"), df3.add_suffix("_m3"), df4.add_suffix("_m4")]
    )
    combined = combined.reindex(sorted(combined.columns), axis=1)

    # Reorder features
    combined = combined[
        [
            "distance_next_m1",
            "distance_next_m2",
            "distance_next_m3",
            "distance_next_m4",
            "vel_magnitude_m1",
            "vel_magnitude_m2",
            "vel_magnitude_m3",
            "vel_magnitude_m4",
            "acc_magnitude_m1",
            "acc_magnitude_m2",
            "acc_magnitude_m3",
            "acc_magnitude_m4",
            "entropy_m1",
            "entropy_m2",
            "entropy_m3",
            "entropy_m4",
            "distance_centroid_static_m1",
            "distance_centroid_static_m2",
            "distance_centroid_static_m3",
            "distance_centroid_static_m4",
        ]
    ]

    # Map
    mapping = {
        "distance_next_m1": r"$d_{\mathrm{next}} \,\, (O)$",
        "distance_next_m2": r"$d_{\mathrm{next}} \,\, (G)$",
        "distance_next_m3": r"$d_{\mathrm{next}} \,\, (Q)$",
        "distance_next_m4": r"$d_{\mathrm{next}} \,\, (F)$",
        "vel_magnitude_m1": r"$\|\mathbf{v}\| \,\, (O)$",
        "vel_magnitude_m2": r"$\|\mathbf{v}\| \,\, (G)$",
        "vel_magnitude_m3": r"$\|\mathbf{v}\| \,\, (Q)$",
        "vel_magnitude_m4": r"$\|\mathbf{v}\| \,\, (F)$",
        "acc_magnitude_m1": r"$\|\mathbf{a}\| \,\, (O)$",
        "acc_magnitude_m2": r"$\|\mathbf{a}\| \,\, (G)$",
        "acc_magnitude_m3": r"$\|\mathbf{a}\| \,\, (Q)$",
        "acc_magnitude_m4": r"$\|\mathbf{a}\| \,\, (F)$",
        "entropy_m1": r"$H \,\, (O)$",
        "entropy_m2": r"$H \,\, (G)$",
        "entropy_m3": r"$H \,\, (Q)$",
        "entropy_m4": r"$H \,\, (F)$",
        "distance_centroid_static_m1": r"$d_{\mathrm{cent}} \,\, (O)$",
        "distance_centroid_static_m2": r"$d_{\mathrm{cent}} \,\, (G)$",
        "distance_centroid_static_m3": r"$d_{\mathrm{cent}} \,\, (Q)$",
        "distance_centroid_static_m4": r"$d_{\mathrm{cent}} \,\, (F)$",
    }

    combined = combined.rename(columns=mapping)
    corrs.append(combined.corr())

fig, axes = plt.subplots(
    1, len(corrs), figsize=(5 * len(corrs), 5), sharex=False, sharey=False
)
vmin, vmax = -1, 1  # fix scale across all
cmap = "coolwarm"  # diverging colormap

# Plot each heatmap without its own colorbar
for ax, corr, name in zip(axes, corrs, files):
    hm = sns.heatmap(
        corr,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        annot=False,
        square=True,
        cbar=False,
        ax=ax,
    )
    ax.set_title(name)

    if ax != axes[0]:
        ax.set_yticklabels([])

# Map titles
mapping = {
    "parkinson": "Neurodegenerative",
    "swear-fluency": "Swear Fluency",
    "italian": "Italian",
    "german": "German",
}
for ax in axes:
    ax.set_title(mapping.get(ax.get_title(), ax.get_title()))

# Add one common horizontal colorbar
# cax = fig.add_axes([0.25, -0.2, 0.5, 0.03])
cax = fig.add_axes([0.25, -0.1, 0.5, 0.03])
fig.colorbar(
    hm.collections[0],
    cax=cax,
    orientation="horizontal",
    fraction=0.1,
    pad=0.1,
)

plt.subplots_adjust(wspace=0.05)

# Save image
DST = "../results/"
plt.savefig(os.path.join(DST, "correlation-heatmaps.pdf"), dpi=300, bbox_inches="tight")

## Experimental

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Thresholds matching your color scale
THRESHOLDS = [
    (1e-4, 4),  # ****
    (1e-3, 3),  # ***
    (1e-2, 2),  # **
    (5e-2, 1),  # *
]
ALPHA = 0.01  # for the binary "significant" count


def count_sig_and_weighted_from_csv(path: Path, alpha: float = ALPHA):
    """
    Returns:
        weighted_score (int): sum of significance weights (0–4)
        binary_count (int): number of pairs where p < alpha
    """
    df = pd.read_csv(path, index_col=0)
    vals = df.values

    # Use upper triangle without diagonal
    mask = np.triu(np.ones_like(vals, dtype=bool), k=1)
    pvals = vals[mask]

    # ---- Binary count ----
    binary_count = int((pvals < alpha).sum())

    # ---- Weighted significance score (0–4) ----
    weighted_total = 0
    for p in pvals:
        weight = 0
        for thr, score in THRESHOLDS:
            if p < thr:
                weight = score
                break
        weighted_total += weight

    return weighted_total, binary_count


import numpy as np
import pandas as pd
import itertools


def cohens_d(x: np.ndarray, y: np.ndarray, correction: bool = True) -> float:
    """
    Calculates Cohen's d.
    If correction=True, returns Hedges' g (corrected for small sample bias).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan

    vx, vy = x.var(ddof=1), y.var(ddof=1)
    pooled_var = ((nx - 1) * vx + (ny - 1) * vy) / (nx + ny - 2)

    if pooled_var <= 0:
        return np.nan

    d = (x.mean() - y.mean()) / np.sqrt(pooled_var)

    if correction:
        # Apply Hedges' g correction
        j_factor = 1 - (3 / (4 * (nx + ny) - 9))
        return d * j_factor

    return d


def mean_effect_size_for_metric(
    df: pd.DataFrame,
    metric_col: str,
    category_col: str = "category",
) -> float:
    """
    Mean |Cohen's d| over all category pairs for a single metric.
    """
    if category_col not in df.columns:
        category_col = "concept"  # For swear words dataset

    cats = df[category_col].dropna().unique()
    pairs = list(itertools.combinations(cats, 2))

    es = []
    for c1, c2 in pairs:
        g1 = df.loc[df[category_col] == c1, metric_col].dropna()
        g2 = df.loc[df[category_col] == c2, metric_col].dropna()
        d = cohens_d(g1.values, g2.values)
        if not np.isnan(d):
            es.append(abs(d))

    return float(np.mean(es)) if es else np.nan

In [ ]:
BASE = Path("../results")

# Map folder names -> pretty labels for the table
MODELS = {
    "openai": "openai",
    "openai-noncum": "openai-noncum",
    "gemini": "gemini",
    "gemini-noncum": "gemini-noncum",
    "qwen": "qwen",
    "qwen-noncum": "qwen-noncum",
    "fasttext-cum": "fasttext",
    "fasttext": "fasttext-noncum",
}

DATASETS = {
    "Neurodegenerative": "parkinson",  # <-- adjust to real folder
    "Swear Words": "swear-fluency",
    "Italian": "italian",
    "German": "german",
}

METRIC_FILES = [
    "pvalues-acc.csv",
    "pvalues-d_centroid.csv",
    "pvalues-d_next.csv",
    "pvalues-entropy.csv",
    "pvalues-vel.csv",
]

METRIC_COLS = [
    "d_next",
    "vel",
    "acc",
    "entropy",
    "d_centroid",
]

In [ ]:
rows = []

for model_key, model_label in MODELS.items():
    row = {"Model": model_label}

    for ds_label, ds_folder in DATASETS.items():
        ds_dir = BASE / model_key / ds_folder

        # ---- significance scores from p-value CSVs ----
        weighted_scores = []
        binary_scores = []

        for fname in METRIC_FILES:
            path = ds_dir / fname
            if path.exists():
                weighted, binary = count_sig_and_weighted_from_csv(path)
                weighted_scores.append(weighted)
                binary_scores.append(binary)

        total_weighted = sum(weighted_scores)
        total_binary = sum(binary_scores)

        # ---- mean effect size from metrics.csv ----
        metrics_path = ds_dir / "metrics.csv"
        if metrics_path.exists():
            mdf = pd.read_csv(metrics_path)
            es_list = [
                mean_effect_size_for_metric(mdf, col)
                for col in METRIC_COLS
                if col in mdf.columns
            ]
            mean_es = float(np.nanmean(es_list)) if es_list else np.nan
        else:
            mean_es = np.nan

        # Store something like: "42 (10), d=0.78"
        row[ds_label] = f"{total_weighted} ({total_binary}), d={mean_es:.2f}"
        # row[ds_label] = f"{total_weighted} ({total_binary})"

    rows.append(row)

# (0.2) is considered small, (0.5) is medium, and (0.8) or higher is large
import tabulate

summary_df = pd.DataFrame(rows)
print(tabulate.tabulate(summary_df, headers="keys", showindex=False))